# Занятие 9. Дескрипторы и сопоставление

Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026
Лекция-опора: L9 «Дескрипторы и сопоставление признаков» · 90 минут в классе (последние 20 — клиника ДЗ 2) · ведёт ассистент

---

Сегодня извлечём **SIFT**-признаки и покажем числом, почему они инвариантны к масштабу и повороту. Потом реализуем **Brute-Force Matcher** с **ratio test** Лоу и проверим, что он отфильтровывает неоднозначные совпадения. В третьей части настроим **FLANN**-матчер для бинарных дескрипторов (ORB) и сравним скорость с BF. В четвёртой — применим **геометрическую верификацию** через `cv2.findHomography` + RANSAC, чтобы отсеять ложные совпадения.

Заготовка начинается на слайдах 12–16 и 49–51 лекции («SIFT descriptor» и «Nearest Neighbor Distance Ratio»).

| Минуты | Что происходит |
|--------|----------------|
| 0–12 | Ассистент разбирает опорный пример: SIFT-ключевые точки и дескрипторы |
| 12–58 | Вы делаете TODO 1–4 |
| 58–67 | Разбор решения |
| 67–87 | Клиника: ДЗ 2 «Сканер документов» — финальные вопросы и связь с L9 |
| 87–90 | Зачёт |

Ничего сдавать не нужно: зачёт ставится в классе по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.
**Занятие 10 — гомография: оценивание и выравнивание.**

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt (docs/setup-guide.md)")

import os, time, timeit
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Фото здания ИУ `lectures/L09-descriptors/demo/iu-building.jpg` (1920 × 1279) читается в сером;
кроп лестницы 256 × 256 тот же, что в занятиях 5–8. Для сопоставления используются два кропа
одного и того же региона: исходный и повёрнутый на 30° + масштабированный ×1.3.

Источники по порядку: репозиторий рядом → скачать с GitHub (Colab) → синтетическая сцена.

In [ ]:
LECTURE_DIR = "lectures/L09-descriptors/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,
                  "../../../iu-intro-cv/" + LECTURE_DIR,
                  "data"]
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILE = "iu-building.jpg"
CROP = (slice(700, 956), slice(1100, 1356))


def synthetic_photo(h=1279, w=1920):
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    img = 120 + 60 * np.sin(xx / 97.0) * np.cos(yy / 71.0) + 0.03 * xx
    for cx, cy, r in ((500, 400, 220), (1400, 900, 300), (1000, 250, 120)):
        img[np.hypot(xx - cx, yy - cy) < r] += 50
    img[700:956:16, 1100:1356] -= 70
    img[700:956, 1100:1356:32] -= 50
    return np.uint8(np.clip(img, 0, 255))


def load_photo():
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, FILE)
        if os.path.exists(p):
            return cio.imread(p, "gray"), d
    try:
        import urllib.request
        os.makedirs("data", exist_ok=True)
        p = os.path.join("data", FILE)
        urllib.request.urlretrieve(RAW_URL + FILE, p)
        return cio.imread(p, "gray"), RAW_URL
    except Exception as e:
        print("файл лекции не скачался (%s) — синтетическая сцена" % type(e).__name__)
        return synthetic_photo(), "синтетика"


photo, SRC = load_photo()
assert photo.shape == (1279, 1920) and photo.dtype == np.uint8
crop1 = photo[CROP].astype(np.float32)
h, w = crop1.shape[:2]
M_transform = cv2.getRotationMatrix2D((w / 2, h / 2), 30, 1.3)
crop2 = cv2.warpAffine(crop1, M_transform, (w, h), flags=cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_REPLICATE)
print("источник:", SRC, "· кадр", photo.shape, photo.dtype, "· кроп", crop1.shape)
viz.grid({"кроп 1 (исходный)": crop1 / 255, "кроп 2 (поворот 30°, ×1.3)": crop2 / 255}, cols=2, size=3.4)

## 2. Опорный пример — разбирает ассистент

Здесь всё написано, задача — **понять каждую строку**.

**SIFT** (слайды 12–16): детектор находит ключевые точки как экстремумы DoG в пирамиде масштабов;
каждой точке присваивается ориентация по гистограмме градиентов; дескриптор — 128-мерный вектор
гистограмм ориентаций градиентов в окрестности 4×4, инвариантный к повороту и масштабу.

**cv2.SIFT_create()** возвращает объект, у которого есть методы `detect()`, `compute()` и
`detectAndCompute()`. Последний — один вызов вместо двух.

In [ ]:
sift = cv2.SIFT_create(nfeatures=500, nOctaveLayers=3, contrastThreshold=0.04,
                       edgeThreshold=10, sigma=1.6)
kp1, desc1 = sift.detectAndCompute(crop1, None)
kp2, desc2 = sift.detectAndCompute(crop2, None)
print("SIFT: кроп 1 — %d точек, дескриптор %s" % (len(kp1), desc1.shape))
print("SIFT: кроп 2 — %d точек, дескриптор %s" % (len(kp2), desc2.shape))
vis1 = cv2.drawKeypoints(crop1, kp1[:50], None, (0, 255, 0),
                         cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis2 = cv2.drawKeypoints(crop2, kp2[:50], None, (0, 255, 0),
                         cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
viz.grid({"SIFT keypoints (кроп 1)": vis1 / 255, "SIFT keypoints (кроп 2)": vis2 / 255}, cols=2, size=3.4)

### Вопрос, на который отвечаем вслух

Почему SIFT-дескриптор имеет размерность 128, а не 32 или 512?
(Ответ — слайд 16: 4×4 сетка подокон × 8 бинов гистограммы ориентаций = 128; меньше — теряется
различимость, больше — растёт память и время сопоставления без существенного выигрыша.)

---

## TODO 1 — Brute-Force Matcher с ratio test (≈ 12 минут)

Слайды 49–51. **Ratio test** Лоу: для каждого дескриптора из первого набора находим двух ближайших
соседей во втором наборе; если отношение расстояний `d1/d2 < ratio_threshold`, совпадение принимается.

Напишите `bf_match_with_ratio(desc1, desc2, ratio=0.75)` → список `cv2.DMatch`.

Скелет — четыре шага:

1. создать `cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)`;
2. `matcher.knnMatch(desc1, desc2, k=2)`;
3. для каждого совпадения: если `m.distance < ratio * n.distance`, добавить `m`;
4. вернуть отфильтрованный список.

> **Подсказка.** `knnMatch` возвращает список списков по 2 элемента; `m, n = match[0], match[1]`.

In [ ]:
def bf_match_with_ratio(desc1, desc2, ratio=0.75):
    """Brute-Force Matcher с ratio test Лоу → список cv2.DMatch."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 6–8 строк: BFMatcher(NORM_L2); knnMatch(k=2); цикл: m.distance < ratio * n.distance; return
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────


good_matches = bf_match_with_ratio(desc1, desc2, ratio=0.75)
assert isinstance(good_matches, list), "результат должен быть списком"
assert all(isinstance(m, cv2.DMatch) for m in good_matches), "элементы списка должны быть cv2.DMatch"
assert len(good_matches) > 0, "должно быть хотя бы одно совпадение после ratio test"
assert len(good_matches) < len(kp1), "ratio test должен отфильтровать часть совпадений"
all_matches = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False).knnMatch(desc1, desc2, k=2)
all_good = sum(1 for m in all_matches if len(m) == 2 and m[0].distance < 1.0 * m[1].distance)
assert len(good_matches) < all_good, "ratio=0.75 должно давать меньше совпадений, чем ratio=1.0"
print("TODO 1 ✔ · совпадений после ratio test: %d из %d возможных" % (len(good_matches), len(kp1)))
vis = cv2.drawMatches(crop1, kp1, crop2, kp2, good_matches[:30], None,
                      matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                      flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
viz.grid({"BF + ratio test (первые 30)": vis / 255}, cols=1, size=5)

### Ratio test в числах (слайд 50)

Чем строже порог ratio, тем меньше ложных совпадений, но и меньше истинных.

In [ ]:
for ratio in (0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0):
    matches = bf_match_with_ratio(desc1, desc2, ratio=ratio)
    print("ratio=%.2f: %d совпадений" % (ratio, len(matches)))

---

## TODO 2 — ORB + FLANN для бинарных дескрипторов (≈ 12 минут)

Слайды 39–41, 55. **ORB** — бинарный дескриптор (256 бит), сравнивается через **Hamming distance**.
**FLANN** быстрее BF на больших наборах.

Напишите `orb_flann_match(img1, img2, ratio=0.75)` → `(kp1, kp2, good_matches)`.

Скелет — пять шагов:

1. `cv2.ORB_create(nfeatures=1000)` → `detectAndCompute`;
2. FLANN index params: `algorithm=FLANN_INDEX_LSH`, `table_number=6`, `key_size=12`, `multi_probe_level=1`;
3. search params: `checks=50`;
4. `cv2.FlannBasedMatcher(index_params, search_params).knnMatch(desc1, desc2, k=2)`;
5. ratio test.

> **Подсказка.** Для бинарных дескрипторов нужен `FLANN_INDEX_LSH`, а не `FLANN_INDEX_KDTREE`.

In [ ]:
def orb_flann_match(img1, img2, ratio=0.75):
    """ORB + FLANN с ratio test → (kp1, kp2, good_matches)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10–14 строк: ORB_create; detectAndCompute; FLANN_INDEX_LSH; FlannBasedMatcher; knnMatch; ratio test
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────


kp1_orb, kp2_orb, good_orb = orb_flann_match(crop1, crop2, ratio=0.75)
assert len(kp1_orb) > 0 and len(kp2_orb) > 0, "ORB должен найти ключевые точки"
assert len(good_orb) > 0, "FLANN + ratio test должен дать хотя бы одно совпадение"
print("TODO 2 ✔ · ORB: %d / %d точек · FLANN + ratio: %d совпадений" % (
    len(kp1_orb), len(kp2_orb), len(good_orb)))
ms = lambda fn, repeat=3: 1e3 * min(timeit.repeat(fn, number=1, repeat=repeat))
orb = cv2.ORB_create(nfeatures=1000)
_, d1 = orb.detectAndCompute(crop1, None)
_, d2 = orb.detectAndCompute(crop2, None)
bf_time = ms(lambda: cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False).knnMatch(d1, d2, k=2))
flann_time = ms(lambda: cv2.FlannBasedMatcher(
    dict(algorithm=cv2.FLANN_INDEX_LSH, table_number=6, key_size=12, multi_probe_level=1),
    dict(checks=50)).knnMatch(d1, d2, k=2))
print("BF: %.1f мс · FLANN: %.1f мс · ускорение ×%.1f" % (bf_time, flann_time, bf_time / max(flann_time, 0.01)))

### Вопрос, на который отвечаем вслух

Почему для ORB нужен `FLANN_INDEX_LSH`, а для SIFT — `FLANN_INDEX_KDTREE`?
(Ответ — слайд 34: ORB — бинарный дескриптор, расстояние Хэмминга; LSH хеширует битовые строки.
SIFT — float32 вектор, евклидово расстояние; KDTree строит дерево по числовым измерениям.)

---

## TODO 3 — геометрическая верификация через findHomography + RANSAC (≈ 14 минут)

Слайды 51, 74. **findHomography + RANSAC** находит гомографию по совпадениям и возвращает маску инлайеров.

Напишите `verify_matches(kp1, kp2, matches, ransac_thresh=3.0)` → `(H, mask, inlier_matches)`.

Скелет — четыре шага:

1. извлечь координаты: `src_pts` и `dst_pts` как `(N, 1, 2)` float32;
2. `cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, ransac_thresh)`;
3. `mask.ravel().astype(bool)`;
4. отфильтровать `matches` по маске.

> **Подсказка.** `kp[m.queryIdx].pt` и `kp[m.trainIdx].pt` — координаты `(x, y)`.

In [ ]:
def verify_matches(kp1, kp2, matches, ransac_thresh=3.0):
    """Геометрическая верификация: findHomography + RANSAC → (H, mask, inlier_matches)."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 8–10 строк: src_pts/dst_pts; findHomography(RANSAC); mask; фильтрация matches
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────


H, mask, inlier_matches = verify_matches(kp1, kp2, good_matches, ransac_thresh=3.0)
assert H is not None, "findHomography должна вернуть матрицу гомографии"
assert H.shape == (3, 3), "гомография — матрица 3×3"
assert len(inlier_matches) > 0, "после RANSAC должны остаться инлайеры"
assert len(inlier_matches) <= len(good_matches), "RANSAC не может добавить совпадения"
print("TODO 3 ✔ · до RANSAC: %d · после RANSAC: %d · инлайеров %.0f%%" % (
    len(good_matches), len(inlier_matches), 100 * len(inlier_matches) / max(len(good_matches), 1)))
vis = cv2.drawMatches(crop1, kp1, crop2, kp2, good_matches[:50], None,
                      matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                      matchesMask=mask[:50] if mask is not None else None,
                      flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
viz.grid({"RANSAC verification (зелёные = инлайеры)": vis / 255}, cols=1, size=5)

### Сколько инлайеров нужно для надёжной гомографии?

Минимум 4 точки для гомографии (8 параметров), но на практике нужно ≥10–15.

In [ ]:
for thresh in (1.0, 3.0, 5.0, 10.0):
    _, _, inl = verify_matches(kp1, kp2, good_matches, ransac_thresh=thresh)
    print("ransac_thresh=%.1f: %d инлайеров" % (thresh, len(inl)))

---

## TODO 4 — полный конвейер: SIFT → BF → ratio → RANSAC (≈ 10 минут)

Объединяем все шаги в одну функцию.

Напишите `match_pipeline(img1, img2, detector='sift', ratio=0.75, ransac_thresh=3.0)` →
`(kp1, kp2, inlier_matches, H)`.

Скелет — три шага:

1. выбрать детектор: `'sift'` → `cv2.SIFT_create()`, `'orb'` → `cv2.ORB_create()`;
2. `detectAndCompute` + matching + ratio test;
3. `verify_matches`.

> **Подсказка.** Для SIFT — `cv2.NORM_L2`, для ORB — `cv2.NORM_HAMMING`.

In [ ]:
def match_pipeline(img1, img2, detector='sift', ratio=0.75, ransac_thresh=3.0):
    """Полный конвейер: детектор → дескриптор → matching → ratio → RANSAC."""
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 12–16 строк: выбор детектора; detectAndCompute; BF/FLANN; ratio test; verify_matches
    raise NotImplementedError("TODO 4")
    # ───────────────────────────────────────────────────────────


kp1_p, kp2_p, inl_p, H_p = match_pipeline(crop1, crop2, detector='sift', ratio=0.75)
assert H_p is not None and H_p.shape == (3, 3), "конвейер должен вернуть гомографию"
assert len(inl_p) >= 4, "для гомографии нужно минимум 4 инлайнера"
print("TODO 4 ✔ · SIFT pipeline: %d инлайеров" % len(inl_p))
kp1_o, kp2_o, inl_o, H_o = match_pipeline(crop1, crop2, detector='orb', ratio=0.75)
print("ORB pipeline: %d инлайеров" % len(inl_o))
vis = cv2.drawMatches(crop1, kp1_p, crop2, kp2_p, inl_p[:30], None,
                      matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                      flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
viz.grid({"Финальные инлайеры (SIFT + RANSAC)": vis / 255}, cols=1, size=5)

### Вопрос, на который отвечаем вслух

Почему нельзя использовать только ratio test без геометрической верификации?
(Ответ — слайд 51: ratio test фильтрует неоднозначные дескрипторы, но не проверяет геометрическую
согласованность. На повторяющихся текстурах ratio test пропускает ложные совпадения.)

---

## Клиника: ДЗ 2 «Сканер документов» — финальные вопросы (последние 20 минут)

**ДЗ 2 «Сканер документов»** выдано на занятии 7 (вт 13.10). Дедлайн уточнить у лектора.
Что из занятия 9 туда идёт:

| Тема занятия 9 | Где в ДЗ 2 | Правило |
|----------------|------------|---------|
| SIFT / ORB | поиск углов документа при плохом освещении | если `approxPolyDP` не находит 4 вершины |
| `findHomography` + RANSAC | выравнивание документа | робастная оценка гомографии |
| Ratio test | фильтрация ложных совпадений | `ratio=0.7–0.8` |
| FLANN | ускорение сопоставления | `FLANN_INDEX_LSH` для ORB |

**Чек-лист перед сдачей ДЗ 2:**

- `pytest tests -q`;
- 4 угла найдены, гомография оценена, документ выпрямлен;
- `uint8`: все операции во `float32`, обратно через `np.clip`.

### `cv2.Stitcher`: автоматическая панорама (демонстрация)

In [ ]:
stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA)
status, pano = stitcher.stitch([crop1.astype(np.uint8), crop2.astype(np.uint8)])
if status == cv2.Stitcher_OK:
    print("Stitcher OK: панорама %d × %d" % (pano.shape[1], pano.shape[0]))
    viz.grid({"Панорама (Stitcher)": pano / 255}, cols=1, size=5)
else:
    print("Stitcher failed: status=%d" % status)

### Вопросы клиники

Принесли вопрос по ДЗ 2 — задавайте сейчас. С занятия 5 ассистент принимает **показ ДЗ**:
одно ДЗ за семестр, две минуты — запуск команды и объяснение одного места кода.

---

## ⭐ Звёздочка (по желанию, +0.5, до конца следующей недели)

**Сравнение детекторов на инвариантность.** Напишите `compare_detectors(img, angles, scales)` →
таблица повторяемости для SIFT, ORB и Harris при разных поворотах и масштабах.

Сдаётся ноутбук с функцией, таблицей и одной фразой: какой детектор лучше для панорам, а какой — для трекинга.

In [ ]:
def compare_detectors(img, angles=(0, 15, 30, 45), scales=(1.0, 1.2, 1.5)):
    """Сравнение повторяемости детекторов при повороте и масштабе."""
    raise NotImplementedError("⭐ звёздочка: сравнение детекторов")


try:
    table = compare_detectors(crop1)
except NotImplementedError:
    print("⭐ не заполнена — это необязательная часть")
else:
    print("⭐ таблица повторяемости:")
    print(table)

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–4 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить, **зачем нужен ratio test** и почему **geometric verification** необходима;
- [ ] на вопросы «почему SIFT инвариантен к масштабу», «чем FLANN отличается от BF» и «сколько точек нужно для гомографии» есть внятный ответ.

2 % итоговой оценки, ставится в классе. Досылать ничего не нужно. **Занятие 10 — гомография: оценивание и выравнивание.**